<a href="https://colab.research.google.com/github/juanpablogm12/langchain/blob/main/langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chat Models

How to use langchain chat models

In [ ]:
pip install -qU "langchain[google-genai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai")


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Create a debate between a {profession_1} teacher and a {profession_2} teacher. response in spanish"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("system", system_template), ("user", "{text}")]
)

prompt = prompt_template.invoke({"profession_1": "physics", "profession_2": "political sciences", "text": "war between israel and palestine"})
prompt.to_messages()

response = model.invoke(prompt)
print(response.content)

¡Excelente tema para un debate! Aquí tienes una confrontación de ideas entre una profesora de física y un profesor de ciencias políticas sobre el conflicto Israel-Palestina.

---

**Moderador/a:** ¡Buenas tardes a todos! Les damos la bienvenida a nuestro espacio de debate. Hoy abordaremos uno de los conflictos más complejos y dolorosos de nuestro tiempo: la guerra entre Israel y Palestina. Para ello, tenemos el privilegio de contar con dos mentes brillantes, cada una desde su particular disciplina.

Por un lado, la **Dra. Elara Ríos**, catedrática de Física Teórica, quien nos ofrecerá una perspectiva basada en sistemas, causas y efectos, y la búsqueda de equilibrios.

Y por el otro, el **Profesor Javier Morales**, especialista en Relaciones Internacionales y Teoría Política, quien nos guiará a través de la historia, la geopolítica y las complejidades de la soberanía y la identidad.

Comencemos con unas palabras iniciales de cada uno. Dra. Ríos, ¿cómo ve usted este conflicto desde la óp

In [ ]:
messages = [
    SystemMessage(content="Translate the following from English into german"),
    HumanMessage(content="i like beer")
]

model.invoke(messages)



AIMessage(content='Ich mag Bier.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--73c15993-8e31-4349-aa14-5622f5d159fa-0', usage_metadata={'input_tokens': 11, 'output_tokens': 147, 'total_tokens': 158, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 143}})

# Semantic Search Engine

In [12]:
pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
pip install langchain-community pypdf

In [4]:
from google.colab import files

uploaded = files.upload()

Saving nke-10k-2023.pdf to nke-10k-2023.pdf


In [11]:
import os
from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')


document loaders

In [7]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))
print(f"{docs[0].page_content[:500]}\n")
print(docs[0].metadata)

107
Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE FISCAL YEAR ENDED MAY 31, 2023
OR
☐  TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE TRANSITION PERIOD FROM                         TO                         .
Commission File No. 1-10635
NIKE, Inc.
(Exact name of Registrant as specified in it

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': 'nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}


text splitter to create chunks

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

516

Generated vectors with an embeddings model


In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [26]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generaeted vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generaeted vectors of length 3072

[-0.0014180047437548637, 0.0006364254513755441, 0.0023240740410983562, -0.028028815984725952, -0.006018347572535276, 0.007032659370452166, 0.008973612450063229, -0.009234011173248291, -0.0018726892303675413, 0.02026733197271824]


Create Vector store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

ids = vector_store.add_documents(documents=all_splits)

results = vector_store.similarity_search("how many distribution centers does Nike have in the US?")

print(results[0])